In [9]:
import requests
import bs4
from bs4 import BeautifulSoup
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [85]:
#### Lists for storing the extracted values
brand, model, color, storage = [], [], [], []
ratings, price = [], []
ram_val=[]
display_size, display_type = [], []
back_camera, front_camera = [], []
battery=[]
processor=[]
warranty = []

for i in range(1,25):
        
    url = "https://www.flipkart.com/search?q=google%20pixel%2010&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off"+str(i)
    
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
    
        products = soup.find_all("div", class_="ZFwe0M row") ### Considering the container as a whole
    
        for prod in products:
    
            # ----- PRODUCT NAME -----
            title_tag = prod.find("div", class_="RG5Slk")
            if title_tag:
                name_text = title_tag.text
                parts = re.split("[^\w\s]", name_text)
    
                brand.append(parts[0].split()[0])
                model.append(" ".join(parts[0].split()[1:]))
                color.append(parts[-3] if len(parts) >= 3 else "NA")
                storage.append(parts[-2] if len(parts) >= 2 else "NA")
            else:
                brand.append("NA")
                model.append("NA")
                coldfor.append("NA")
                storage.append("NA")
    
            # ----- RATING -----
            rating_tag = prod.find("div", class_="MKiFS6")
            ratings.append(rating_tag.text if rating_tag else "NA")
    
            # ----- PRICE -----
            price_tag = prod.find("div", class_="hZ3P6w DeU9vF")
            price.append(re.sub("[^\d]", "", price_tag.text) if price_tag else "NA")
    
            # ----- SPECS -----
            specs = prod.find_all("li", class_="DTBslk")
    
            ram_value="NA"
            disp_size = "NA"
            back_cam = front_cam = "NA"
            chip_name = "NA"
            processor_value = "NA"
            warranty_value = "NA"

    
            for s in specs:
                text = s.text.lower()
    
                #RAM
                # RAM
                if "gb" in text:
                    match = re.search(r"(\d+(?:\.\d+)?)\s*GB", text, re.IGNORECASE)
                    if match:
                        ram_value = match.group(1)

                # Display
                if "inch" in text:
                    size_match = re.search(r'(\d+(?:\.\d+)?)\s*inch', s.text, re.IGNORECASE)
                    if size_match:
                        disp_size = size_match.group(1)
                    disp_type = s.text.split(")")[-1].strip()

    
                # Camera
                if "mp" in text:
                    cams = re.findall(r"\d+mp", text)
                    if len(cams) >= 2:
                        back_cam, front_cam = cams[0], cams[-1]
    
                # Chip
                if "processor" in text:
                    proc_match = re.search(r'(Tensor\s*G\d+|Tensor)', s.text, re.IGNORECASE)
                    if proc_match:
                        processor_value = proc_match.group(1).title()

                # Warranty
                if "warranty" in text:
                    warranty_value = " ".join(s.text.split()[:2])
    
            ram_val.append(ram_value)
            display_size.append(disp_size)
            display_type.append(disp_type)
            back_camera.append(back_cam)
            front_camera.append(front_cam)
            processor.append(processor_value)
            warranty.append(warranty_value)

    else:
        print("Failed to fetch page:",i, response.status_code)


In [86]:
df = pd.DataFrame({
    "Brand": brand,
    "Model": model,
    "Color": color,
    "Storage": storage,
    "Rating": ratings,
    "Price": price,
    "Ram": ram_value,
    "Display Size": display_size,
    "Display Type": display_type,
    "Back Camera": back_camera,
    "Front Camera": front_camera,
    "Processor": processor,
    "Warranty": warranty
})

In [87]:
df.head(15)

,Brand,Model,Color,Storage,Rating,Price,Ram,Display Size,Display Type,Back Camera,Front Camera,Processor,Warranty
0,Google,Pixel 10,Frost,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
1,Google,Pixel 10,Lemongrass,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
2,Google,Pixel 10,Obsidian,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
3,Google,Pixel 10,Indigo,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
4,Google,Pixel 10 Pro XL,Obsidian,256 GB,4.6,124999,8,6.8,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
5,Google,Pixel 10 Pro XL,Jade,256 GB,4.6,124999,8,6.8,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
6,Google,Pixel 10 Pro,Obsidian,256 GB,4.6,109999,8,6.3,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
7,Google,Pixel 10 Pro XL,Moonstone,256 GB,4.6,124999,8,6.8,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
8,Google,Pixel 10 Pro,Porcelain,256 GB,4.6,109999,8,6.3,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
9,Google,Pixel 10 Pro,Moonstone,256 GB,4.6,109999,8,6.3,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year


In [88]:
df['Display Type'] = df['Display Type'].replace('Display', 'OLED Display')

In [89]:
df.head(20)

,Brand,Model,Color,Storage,Rating,Price,Ram,Display Size,Display Type,Back Camera,Front Camera,Processor,Warranty
0,Google,Pixel 10,Frost,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
1,Google,Pixel 10,Lemongrass,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
2,Google,Pixel 10,Obsidian,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
3,Google,Pixel 10,Indigo,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
4,Google,Pixel 10 Pro XL,Obsidian,256 GB,4.6,124999,8,6.8,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
5,Google,Pixel 10 Pro XL,Jade,256 GB,4.6,124999,8,6.8,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
6,Google,Pixel 10 Pro,Obsidian,256 GB,4.6,109999,8,6.3,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
7,Google,Pixel 10 Pro XL,Moonstone,256 GB,4.6,124999,8,6.8,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
8,Google,Pixel 10 Pro,Porcelain,256 GB,4.6,109999,8,6.3,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
9,Google,Pixel 10 Pro,Moonstone,256 GB,4.6,109999,8,6.3,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year


In [90]:
mask = df['Back Camera'].str.match(r'^\dmp$', na=False)

df.loc[mask, 'Back Camera'] = df.loc[mask, 'Back Camera'].apply(
    lambda x: "12.2mp" if x == "2mp" else x
)


In [91]:
df.head(20)

,Brand,Model,Color,Storage,Rating,Price,Ram,Display Size,Display Type,Back Camera,Front Camera,Processor,Warranty
0,Google,Pixel 10,Frost,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
1,Google,Pixel 10,Lemongrass,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
2,Google,Pixel 10,Obsidian,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
3,Google,Pixel 10,Indigo,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
4,Google,Pixel 10 Pro XL,Obsidian,256 GB,4.6,124999,8,6.8,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
5,Google,Pixel 10 Pro XL,Jade,256 GB,4.6,124999,8,6.8,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
6,Google,Pixel 10 Pro,Obsidian,256 GB,4.6,109999,8,6.3,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
7,Google,Pixel 10 Pro XL,Moonstone,256 GB,4.6,124999,8,6.8,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
8,Google,Pixel 10 Pro,Porcelain,256 GB,4.6,109999,8,6.3,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
9,Google,Pixel 10 Pro,Moonstone,256 GB,4.6,109999,8,6.3,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year


In [92]:
df = df[(df['Back Camera'] != 'NA') & (df['Front Camera'] != 'NA')].reset_index(drop=True)

In [93]:
df.head(20)

,Brand,Model,Color,Storage,Rating,Price,Ram,Display Size,Display Type,Back Camera,Front Camera,Processor,Warranty
0,Google,Pixel 10,Frost,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
1,Google,Pixel 10,Lemongrass,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
2,Google,Pixel 10,Obsidian,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
3,Google,Pixel 10,Indigo,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
4,Google,Pixel 10 Pro XL,Obsidian,256 GB,4.6,124999,8,6.8,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
5,Google,Pixel 10 Pro XL,Jade,256 GB,4.6,124999,8,6.8,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
6,Google,Pixel 10 Pro,Obsidian,256 GB,4.6,109999,8,6.3,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
7,Google,Pixel 10 Pro XL,Moonstone,256 GB,4.6,124999,8,6.8,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
8,Google,Pixel 10 Pro,Porcelain,256 GB,4.6,109999,8,6.3,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
9,Google,Pixel 10 Pro,Moonstone,256 GB,4.6,109999,8,6.3,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year


In [94]:
df

,Brand,Model,Color,Storage,Rating,Price,Ram,Display Size,Display Type,Back Camera,Front Camera,Processor,Warranty
0,Google,Pixel 10,Frost,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
1,Google,Pixel 10,Lemongrass,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
2,Google,Pixel 10,Obsidian,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
3,Google,Pixel 10,Indigo,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
4,Google,Pixel 10 Pro XL,Obsidian,256 GB,4.6,124999,8,6.8,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
...,...,...,...,...,...,...,...,...,...,...,...,...,...
547,Google,Pixel 6a,Charcoal,128 GB,4.3,43999,8,6.14,Full HD+ Display,12.2mp,8mp,Tensor,1 Year
548,Google,Pixel 8a,Porcelain,128 GB,4.3,49999,8,6.1,Full HD+ Display,64mp,13mp,Tensor G3,1 Year
549,Google,Pixel 9,Obsidian,256 GB,4.6,79999,8,6.3,OLED Display,50mp,5mp,Tensor G4,1 Year
550,Google,Pixel 9,Wintergreen,256 GB,4.6,79999,8,6.3,OLED Display,50mp,5mp,Tensor G4,1 Year


In [100]:
df

,Brand,Model,Color,Storage,Rating,Price,Ram,Display Size,Display Type,Back Camera,Front Camera,Processor,Warranty
0,Google,Pixel 10,Frost,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
1,Google,Pixel 10,Lemongrass,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
2,Google,Pixel 10,Obsidian,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
3,Google,Pixel 10,Indigo,256 GB,4.5,73999,8,6.3,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
4,Google,Pixel 10 Pro XL,Obsidian,256 GB,4.6,124999,8,6.8,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
...,...,...,...,...,...,...,...,...,...,...,...,...,...
547,Google,Pixel 6a,Charcoal,128 GB,4.3,43999,8,6.14,Full HD+ Display,12.2mp,8mp,Tensor,1 Year
548,Google,Pixel 8a,Porcelain,128 GB,4.3,49999,8,6.1,Full HD+ Display,64mp,13mp,Tensor G3,1 Year
549,Google,Pixel 9,Obsidian,256 GB,4.6,79999,8,6.3,OLED Display,50mp,5mp,Tensor G4,1 Year
550,Google,Pixel 9,Wintergreen,256 GB,4.6,79999,8,6.3,OLED Display,50mp,5mp,Tensor G4,1 Year


In [103]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552 entries, 0 to 551
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Brand         552 non-null    object
 1   Model         552 non-null    object
 2   Color         552 non-null    object
 3   Storage       552 non-null    object
 4   Rating        552 non-null    object
 5   Price         552 non-null    object
 6   Ram           552 non-null    object
 7   Display Size  552 non-null    object
 8   Display Type  552 non-null    object
 9   Back Camera   552 non-null    object
 10  Front Camera  552 non-null    object
 11  Processor     552 non-null    object
 12  Warranty      552 non-null    object
dtypes: object(13)
memory usage: 56.2+ KB


In [104]:
df['Ram'] = df['Ram'].astype(int)

In [106]:
df['Price'] = df['Price'].astype(int)

In [107]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552 entries, 0 to 551
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Brand         552 non-null    object
 1   Model         552 non-null    object
 2   Color         552 non-null    object
 3   Storage       552 non-null    object
 4   Rating        552 non-null    object
 5   Price         552 non-null    int64 
 6   Ram           552 non-null    int64 
 7   Display Size  552 non-null    object
 8   Display Type  552 non-null    object
 9   Back Camera   552 non-null    object
 10  Front Camera  552 non-null    object
 11  Processor     552 non-null    object
 12  Warranty      552 non-null    object
dtypes: int64(2), object(11)
memory usage: 56.2+ KB


In [108]:
df['Display Size'] = df['Display Size'].astype(float)
df["Rating"]=df["Rating"].astype(float)

In [109]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552 entries, 0 to 551
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Brand         552 non-null    object 
 1   Model         552 non-null    object 
 2   Color         552 non-null    object 
 3   Storage       552 non-null    object 
 4   Rating        552 non-null    float64
 5   Price         552 non-null    int64  
 6   Ram           552 non-null    int64  
 7   Display Size  552 non-null    float64
 8   Display Type  552 non-null    object 
 9   Back Camera   552 non-null    object 
 10  Front Camera  552 non-null    object 
 11  Processor     552 non-null    object 
 12  Warranty      552 non-null    object 
dtypes: float64(2), int64(2), object(9)
memory usage: 56.2+ KB


In [110]:
df['Storage'] = df['Storage'].str.replace(' GB','')

In [114]:
df['Storage'] = df['Storage'].astype(int)

In [115]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552 entries, 0 to 551
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Brand         552 non-null    object 
 1   Model         552 non-null    object 
 2   Color         552 non-null    object 
 3   Storage       552 non-null    int64  
 4   Rating        552 non-null    float64
 5   Price         552 non-null    int64  
 6   Ram           552 non-null    int64  
 7   Display Size  552 non-null    float64
 8   Display Type  552 non-null    object 
 9   Back Camera   552 non-null    object 
 10  Front Camera  552 non-null    object 
 11  Processor     552 non-null    object 
 12  Warranty      552 non-null    object 
dtypes: float64(2), int64(3), object(8)
memory usage: 56.2+ KB


In [116]:
df

,Brand,Model,Color,Storage,Rating,Price,Ram,Display Size,Display Type,Back Camera,Front Camera,Processor,Warranty
0,Google,Pixel 10,Frost,256,4.5,73999,8,6.30,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
1,Google,Pixel 10,Lemongrass,256,4.5,73999,8,6.30,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
2,Google,Pixel 10,Obsidian,256,4.5,73999,8,6.30,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
3,Google,Pixel 10,Indigo,256,4.5,73999,8,6.30,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
4,Google,Pixel 10 Pro XL,Obsidian,256,4.6,124999,8,6.80,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
...,...,...,...,...,...,...,...,...,...,...,...,...,...
547,Google,Pixel 6a,Charcoal,128,4.3,43999,8,6.14,Full HD+ Display,12.2mp,8mp,Tensor,1 Year
548,Google,Pixel 8a,Porcelain,128,4.3,49999,8,6.10,Full HD+ Display,64mp,13mp,Tensor G3,1 Year
549,Google,Pixel 9,Obsidian,256,4.6,79999,8,6.30,OLED Display,50mp,5mp,Tensor G4,1 Year
550,Google,Pixel 9,Wintergreen,256,4.6,79999,8,6.30,OLED Display,50mp,5mp,Tensor G4,1 Year


In [117]:
df.head(50)

,Brand,Model,Color,Storage,Rating,Price,Ram,Display Size,Display Type,Back Camera,Front Camera,Processor,Warranty
0,Google,Pixel 10,Frost,256,4.5,73999,8,6.300,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
1,Google,Pixel 10,Lemongrass,256,4.5,73999,8,6.300,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
2,Google,Pixel 10,Obsidian,256,4.5,73999,8,6.300,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
3,Google,Pixel 10,Indigo,256,4.5,73999,8,6.300,Quad HD+ Display,48mp,5mp,Tensor G5,1 Year
4,Google,Pixel 10 Pro XL,Obsidian,256,4.6,124999,8,6.800,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
5,Google,Pixel 10 Pro XL,Jade,256,4.6,124999,8,6.800,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
6,Google,Pixel 10 Pro,Obsidian,256,4.6,109999,8,6.300,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
7,Google,Pixel 10 Pro XL,Moonstone,256,4.6,124999,8,6.800,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
8,Google,Pixel 10 Pro,Porcelain,256,4.6,109999,8,6.300,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year
9,Google,Pixel 10 Pro,Moonstone,256,4.6,109999,8,6.300,Quad HD+ Display,48mp,42mp,Tensor G5,1 Year


In [118]:
df.to_csv("pixel_mobiles.csv")